# Amazon Reviews — EDA & Cleaning
End-to-end recommender project — data prep stage

## 1. Load data

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv('amazon_review.csv')
df.head()

,Id,ProductId,UserId,ProfileName,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text
0,1,B001E4KFG0,A3SGXH7AUHU8GW,delmartian,1,1,5,1303862400,Good Quality Dog Food,I have bought several of the Vitality canned d...
1,2,B00813GRG4,A1D87F6ZCVE5NK,dll pa,0,0,1,1346976000,Not as Advertised,Product arrived labeled as Jumbo Salted Peanut...
2,3,B000LQOCH0,ABXLMWJIXXAIN,"Natalia Corres ""Natalia Corres""",1,1,4,1219017600,"""Delight"" says it all",This is a confection that has been around a fe...
3,4,B000UA0QIQ,A395BORC6FGVXV,Karl,3,3,2,1307923200,Cough Medicine,If you are looking for the secret ingredient i...
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,"Michael D. Bigham ""M. Wassir""",0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...


## 2. Basic inspection

In [2]:
print("Shape:", df.shape)
df.info()

Shape: (568454, 10)
<class 'pandas.DataFrame'>
RangeIndex: 568454 entries, 0 to 568453
Data columns (total 10 columns):
 #   Column                  Non-Null Count   Dtype
---  ------                  --------------   -----
 0   Id                      568454 non-null  int64
 1   ProductId               568454 non-null  str  
 2   UserId                  568454 non-null  str  
 3   ProfileName             568428 non-null  str  
 4   HelpfulnessNumerator    568454 non-null  int64
 5   HelpfulnessDenominator  568454 non-null  int64
 6   Score                   568454 non-null  int64
 7   Time                    568454 non-null  int64
 8   Summary                 568427 non-null  str  
 9   Text                    568454 non-null  str  
dtypes: int64(5), str(5)
memory usage: 313.1 MB


In [3]:
df.describe()

,Id,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time
count,568454.000000,568454.000000,568454.00000,568454.000000,5.684540e+05
mean,284227.500000,1.743817,2.22881,4.183199,1.296257e+09
std,164098.679298,7.636513,8.28974,1.310436,4.804331e+07
min,1.000000,0.000000,0.00000,1.000000,9.393408e+08
25%,142114.250000,0.000000,0.00000,4.000000,1.271290e+09
50%,284227.500000,0.000000,1.00000,5.000000,1.311120e+09
75%,426340.750000,2.000000,2.00000,5.000000,1.332720e+09
max,568454.000000,866.000000,923.00000,5.000000,1.351210e+09


In [4]:
df.isnull().sum()

Id                         0
ProductId                  0
UserId                     0
ProfileName               26
HelpfulnessNumerator       0
HelpfulnessDenominator     0
Score                      0
Time                       0
Summary                   27
Text                       0
dtype: int64

## 3. Score distribution
Understand class balance — useful later when validating the sentiment model against `Score`.

In [5]:
df['Score'].value_counts(normalize=True).sort_index()

Score
1    0.091948
2    0.052368
3    0.075010
4    0.141885
5    0.638789
Name: proportion, dtype: float64

## 4. Duplicate check
Full-row duplicates are usually rare. The real issue in this dataset: the same review text posted by the same user under multiple product listings. Check `UserId` + `Text` combo, not just `Text` alone.

In [6]:
print("Full row duplicates:", df.duplicated().sum())
print("Duplicates on UserId+Text:", df.duplicated(subset=['UserId', 'Text']).sum())

Full row duplicates: 0
Duplicates on UserId+Text: 174848


## 5. Product-level distribution
Critical for the recommender — products with very few reviews give a weak signal.

In [7]:
reviews_per_product = df.groupby('ProductId').size()
reviews_per_product.describe()

count    74258.000000
mean         7.655121
std         26.453485
min          1.000000
25%          1.000000
50%          2.000000
75%          5.000000
max        913.000000
dtype: float64

In [8]:
print("Unique products:", df['ProductId'].nunique())
print("Products with only 1 review:", (reviews_per_product == 1).sum())
print("Products with >=3 reviews:", (reviews_per_product >= 3).sum())
print("Products with >=5 reviews:", (reviews_per_product >= 5).sum())

Unique products: 74258
Products with only 1 review: 30408
Products with >=3 reviews: 31588
Products with >=5 reviews: 20415


## 6. Text length check
Very short reviews add little signal for embeddings; very long ones may need capping for the embedding model's token limit.

In [9]:
text_word_count = df['Text'].str.split().str.len()
text_word_count.describe()

count    568454.000000
mean         80.264023
std          79.455384
min           3.000000
25%          33.000000
50%          56.000000
75%          98.000000
max        3432.000000
Name: Text, dtype: float64

## 7. HTML artifact check
This dataset has leftover `<br />` tags in the text.

In [10]:
print("Rows containing HTML tags:", df['Text'].str.contains('<br', na=False).sum())

Rows containing HTML tags: 142924


## 8. Cleaning
Apply everything found above: drop unneeded columns, dedupe, strip HTML, filter short reviews, filter low-review products.

In [11]:
df_clean = df.copy()

# Drop columns not needed for the recommender
df_clean = df_clean.drop(columns=['ProfileName'])

# Dedupe on UserId + Text (real duplicates, not just full-row)
df_clean = df_clean.drop_duplicates(subset=['UserId', 'Text'])

# Strip HTML artifacts
df_clean['Text'] = df_clean['Text'].str.replace(r'<br\s*/?>', ' ', regex=True)

# Recompute word count on cleaned text, filter very short reviews
df_clean['word_count'] = df_clean['Text'].str.split().str.len()
df_clean = df_clean[df_clean['word_count'] >= 5]

# Keep only products with at least 3 reviews
product_counts = df_clean.groupby('ProductId').size()
valid_products = product_counts[product_counts >= 3].index
df_clean = df_clean[df_clean['ProductId'].isin(valid_products)]

print("Rows before:", len(df))
print("Rows after:", len(df_clean))
print("Unique products after:", df_clean['ProductId'].nunique())

Rows before: 568454
Rows after: 342167
Unique products after: 27606


## 9. Final check

In [12]:
df_clean.isnull().sum()

Id                        0
ProductId                 0
UserId                    0
HelpfulnessNumerator      0
HelpfulnessDenominator    0
Score                     0
Time                      0
Summary                   3
Text                      0
word_count                0
dtype: int64

In [13]:
df_clean.head()

,Id,ProductId,UserId,HelpfulnessNumerator,HelpfulnessDenominator,Score,Time,Summary,Text,word_count
4,5,B006K2ZZ7K,A1UQRSCLF8GW1T,0,0,5,1350777600,Great taffy,Great taffy at a great price. There was a wid...,27
5,6,B006K2ZZ7K,ADT0SRK1MGOEU,0,0,4,1342051200,Nice Taffy,I got a wild hair for taffy and ordered this f...,72
6,7,B006K2ZZ7K,A1SP2KVKFXXRU1,0,0,5,1340150400,Great! Just as good as the expensive brands!,This saltwater taffy had great flavors and was...,49
7,8,B006K2ZZ7K,A3JRGQVEQN31IQ,0,0,5,1336003200,"Wonderful, tasty taffy",This taffy is so good. It is very soft and ch...,24
13,14,B001GVISJM,A18ECVX2RJ7HUE,2,2,4,1288915200,fresh and greasy!,good flavor! these came securely packed... the...,15


## 10. Save cleaned data

In [14]:
df_clean.to_parquet('amazon_reviews_clean.parquet', index=False)
print("Saved.")

Saved.


In [15]:
import pandas as pd
import pyarrow as pa

print("pandas:", pd.__version__)
print("pyarrow:", pa.__version__)

pandas: 3.0.5
pyarrow: 25.0.1
